# Faster R-CNN для обнаружения объектов

**Модель:** Faster R-CNN с backbone ResNet50-FPN

**Особенности архитектуры:**
- Region Proposal Network (RPN) для генерации предложений регионов
- ROI Pooling для извлечения признаков из предложенных регионов
- Двухэтапное обнаружение: сначала предложения, потом классификация
- Feature Pyramid Network (FPN) для мультимасштабного обнаружения

**Преимущества:**
- Высокая точность обнаружения
- Хорошо работает с объектами разных размеров
- Стабильное обучение

**Недостатки:**
- Медленнее, чем одноэтапные детекторы (YOLO, SSD)
- Больше параметров модели
- Требует больше памяти GPU

In [ ]:
# Установка необходимых библиотек
!pip install torch torchvision matplotlib pillow numpy -q

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict

print(f'PyTorch версия: {torch.__version__}')
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA устройство: {torch.cuda.get_device_name(0)}')

In [ ]:
# Конфигурация путей к данным
DATA_ROOT = '/content/drive/MyDrive/Датасет'  # Измените на свой путь
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, 'train/images')
TRAIN_LABEL_DIR = os.path.join(DATA_ROOT, 'train/labels')
VAL_IMG_DIR = os.path.join(DATA_ROOT, 'val/images')
VAL_LABEL_DIR = os.path.join(DATA_ROOT, 'val/labels')
TEST_IMG_DIR = os.path.join(DATA_ROOT, 'test/images')
TEST_LABEL_DIR = os.path.join(DATA_ROOT, 'test/labels')

# Гиперпараметры
NUM_CLASSES = 6  # 5 классов + 1 фон (измените под ваш датасет)
BATCH_SIZE = 2
NUM_EPOCHS = 10
LEARNING_RATE = 0.005
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

print(f'Используемое устройство: {DEVICE}')

In [ ]:
class YOLODataset(Dataset):
    """Датасет для загрузки данных в формате YOLO"""
    
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith('.png')]
    
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        # Загрузка изображения
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        img = Image.open(img_path).convert('RGB')
        
        # Загрузка аннотаций (YOLO формат: class x_center y_center width height)
        label_path = os.path.join(self.label_dir, self.imgs[idx].replace('.png', '.txt'))
        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    # Парсинг: class x_center y_center width height
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    
                    # Конвертация из YOLO формата (нормализованные координаты)
                    # в формат R-CNN (пиксельные координаты x_min, y_min, x_max, y_max)
                    img_width, img_height = img.size
                    x_min = (x_center - width / 2) * img_width
                    y_min = (y_center - height / 2) * img_height
                    x_max = (x_center + width / 2) * img_width
                    y_max = (y_center + height / 2) * img_height
                    
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(class_id) + 1)  # +1 потому что 0 зарезервирован для фона
        
        # Конвертация в тензоры
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.tensor([])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': image_id,
            'area': area,
            'iscrowd': iscrowd
        }
        
        # Преобразование изображения в тензор
        img = torchvision.transforms.ToTensor()(img)
        
        return img, target

def collate_fn(batch):
    """Функция для объединения батча (необходима для R-CNN)"""
    return tuple(zip(*batch))

In [ ]:
# Создание датасетов и загрузчиков данных
train_dataset = YOLODataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR)
val_dataset = YOLODataset(VAL_IMG_DIR, VAL_LABEL_DIR)
test_dataset = YOLODataset(TEST_IMG_DIR, TEST_LABEL_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Размер обучающего набора: {len(train_dataset)}')
print(f'Размер валидационного набора: {len(val_dataset)}')
print(f'Размер тестового набора: {len(test_dataset)}')

In [ ]:
# ============================================================
# СОЗДАНИЕ МОДЕЛИ FASTER R-CNN
# ============================================================
#
# Faster R-CNN состоит из:
# 1. Backbone (ResNet50-FPN) - извлечение признаков из изображения
# 2. RPN (Region Proposal Network) - предложение потенциальных регионов с объектами
# 3. ROI Pooling - выравнивание размеров признаков из разных регионов
# 4. Box Head - классификация и уточнение координат bbox
#
# Особенности:
# - Двухэтапный детектор: сначала находит регионы, потом классифицирует
# - FPN (Feature Pyramid Network) для обнаружения объектов разных размеров
# - Более точный, но медленнее одноэтапных детекторов
# ============================================================

# Очистка кэша GPU перед созданием модели
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Загрузка предобученной модели Faster R-CNN с ResNet50-FPN backbone
model = fasterrcnn_resnet50_fpn(pretrained=True)

# Замена головы классификатора под наше количество классов
# in_features - количество входных признаков из backbone
in_features = model.roi_heads.box_predictor.cls_score.in_features

# Создаем новый предиктор с нужным количеством классов
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# Перемещение модели на GPU/CPU
model = model.to(DEVICE)

print(f'Модель создана и перемещена на {DEVICE}')
print(f'Количество классов (включая фон): {NUM_CLASSES}')

In [ ]:
# Настройка оптимизатора и планировщика скорости обучения
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LEARNING_RATE, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

print('Оптимизатор и планировщик настроены')

In [ ]:
# Функция обучения на одной эпохе
def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    total_loss = 0
    
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward pass - модель возвращает словарь с лоссами при обучении
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
    
    return total_loss / len(data_loader)

# Функция валидации
def validate(model, data_loader, device):
    model.train()  # Режим train для получения лоссов
    total_loss = 0
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()
    
    return total_loss / len(data_loader)

In [ ]:
# Обучение модели
print('Начало обучения...')
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # Обучение
    train_loss = train_one_epoch(model, optimizer, train_loader, DEVICE)
    train_losses.append(train_loss)
    
    # Валидация
    val_loss = validate(model, val_loader, DEVICE)
    val_losses.append(val_loss)
    
    # Обновление learning rate
    lr_scheduler.step()
    
    print(f'Эпоха {epoch+1}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'LR: {optimizer.param_groups[0]["lr"]:.6f}')

print('Обучение завершено!')

In [ ]:
# Визуализация графиков обучения с интерпретацией
fig, axes = plt.subplots(1, 1, figsize=(12, 6))

axes.plot(train_losses, label='Train Loss', marker='o', color='blue')
axes.plot(val_losses, label='Val Loss', marker='s', color='red')
axes.set_xlabel('Эпоха', fontsize=12)
axes.set_ylabel('Loss', fontsize=12)
axes.set_title('График обучения Faster R-CNN', fontsize=14, fontweight='bold')
axes.legend(fontsize=11)
axes.grid(True, alpha=0.3)

# Подсказки по интерпретации
axes.text(0.02, 0.98, 
    '💡 Как интерпретировать:\n'
    '✅ ХОРОШО: Обе линии снижаются, близки друг к другу\n'
    '⚠️ ПЕРЕОБУЧЕНИЕ: Синяя падает, красная растет\n'
    '⚠️ НЕДООБУЧЕНИЕ: Обе линии высокие, не снижаются\n'
    '📊 Меньше = Лучше',
    transform=axes.transAxes, fontsize=10,
    verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print(f'Финальный Train Loss: {train_losses[-1]:.4f}')
print(f'Финальный Val Loss: {val_losses[-1]:.4f}')

In [ ]:
# Функция для вычисления IoU (Intersection over Union)
def compute_iou(box1, box2):
    """Вычисление IoU между двумя bbox"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

# Функция для вычисления метрик
def calculate_metrics(model, data_loader, device, iou_threshold=0.5, score_threshold=0.1):
    """Вычисление Precision, Recall, F1-Score"""
    model.eval()
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = list(image.to(device) for image in images)
            predictions = model(images)
            
            for pred, target in zip(predictions, targets):
                # Фильтрация предсказаний по уверенности
                pred_boxes = pred['boxes'][pred['scores'] > score_threshold].cpu().numpy()
                pred_labels = pred['labels'][pred['scores'] > score_threshold].cpu().numpy()
                
                gt_boxes = target['boxes'].cpu().numpy()
                gt_labels = target['labels'].cpu().numpy()
                
                # Подсчет True Positives, False Positives
                matched_gt = set()
                for pred_box, pred_label in zip(pred_boxes, pred_labels):
                    match_found = False
                    for i, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
                        if i not in matched_gt and pred_label == gt_label:
                            iou = compute_iou(pred_box, gt_box)
                            if iou >= iou_threshold:
                                true_positives += 1
                                matched_gt.add(i)
                                match_found = True
                                break
                    if not match_found:
                        false_positives += 1
                
                # False Negatives - пропущенные ground truth объекты
                false_negatives += len(gt_boxes) - len(matched_gt)
    
    # Вычисление метрик
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives
    }

In [ ]:
# Вычисление метрик на тестовом наборе
print('Вычисление метрик на тестовом наборе...')
test_metrics = calculate_metrics(model, test_loader, DEVICE, iou_threshold=0.5, score_threshold=0.1)

print('\n' + '='*50)
print('МЕТРИКИ НА ТЕСТОВОМ НАБОРЕ (Faster R-CNN)')
print('='*50)
print(f'Precision: {test_metrics["precision"]:.4f}')
print(f'Recall: {test_metrics["recall"]:.4f}')
print(f'F1-Score: {test_metrics["f1_score"]:.4f}')
print(f'\nДетали:')
print(f'True Positives: {test_metrics["true_positives"]}')
print(f'False Positives: {test_metrics["false_positives"]}')
print(f'False Negatives: {test_metrics["false_negatives"]}')
print('='*50)

# Интерпретация результатов
print('\n💡 Интерпретация метрик:')
print('Precision (точность) - какой % предсказаний правильный (больше = лучше)')
print('Recall (полнота) - какой % объектов найден (больше = лучше)')
print('F1-Score - баланс между точностью и полнотой (больше = лучше)')

In [ ]:
# Визуализация метрик
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

metrics_names = ['Precision', 'Recall', 'F1-Score']
metrics_values = [test_metrics['precision'], test_metrics['recall'], test_metrics['f1_score']]
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Значение', fontsize=12)
ax.set_title('Метрики качества Faster R-CNN на тестовом наборе', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Добавление значений на столбцы
for bar, value in zip(bars, metrics_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Подсказка
ax.text(0.5, 0.95, '📊 Больше = Лучше (максимум = 1.0)',
        transform=ax.transAxes, fontsize=11,
        ha='center', va='top',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация предсказаний на тестовых изображениях
model.eval()

# Выбираем 9 случайных изображений из тестового набора
indices = np.random.choice(len(test_dataset), min(9, len(test_dataset)), replace=False)

fig, axes = plt.subplots(3, 3, figsize=(20, 20))
axes = axes.flatten()

with torch.no_grad():
    for idx, ax in zip(indices, axes):
        img, target = test_dataset[idx]
        prediction = model([img.to(DEVICE)])[0]
        
        # Конвертация изображения для отображения
        img_np = img.permute(1, 2, 0).cpu().numpy()
        
        ax.imshow(img_np)
        ax.axis('off')
        
        # Отрисовка Ground Truth (зеленый)
        for box, label in zip(target['boxes'], target['labels']):
            x1, y1, x2, y2 = box.cpu().numpy()
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=2, edgecolor='green',
                                     facecolor='none', label='GT')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f'GT: {label.item()-1}',
                   color='green', fontsize=10, weight='bold',
                   bbox=dict(facecolor='white', alpha=0.7))
        
        # Отрисовка предсказаний (красный) с порогом 0.1
        for box, label, score in zip(prediction['boxes'], prediction['labels'], prediction['scores']):
            if score > 0.1:  # Порог уверенности
                x1, y1, x2, y2 = box.cpu().numpy()
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                         linewidth=2, edgecolor='red',
                                         facecolor='none', linestyle='--')
                ax.add_patch(rect)
                ax.text(x2, y2+5, f'Pred: {label.item()-1} ({score:.2f})',
                       color='red', fontsize=10, weight='bold',
                       bbox=dict(facecolor='white', alpha=0.7))
        
        ax.set_title(f'Изображение {idx}', fontsize=12)

plt.suptitle('Результаты обнаружения Faster R-CNN\n(Зеленый=Ground Truth, Красный=Предсказание)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Визуализация завершена!')

## Заключение

**Faster R-CNN** - это мощный двухэтапный детектор объектов:

**Когда использовать:**
- Когда нужна высокая точность обнаружения
- Когда скорость inference не критична
- Для обнаружения объектов разных размеров
- Когда есть достаточно GPU памяти

**Когда НЕ использовать:**
- Для real-time приложений (используйте YOLO)
- При ограниченной GPU памяти (используйте SSD)
- Когда важна скорость обработки